# Respondendo às perguntas de negócio

Seis perguntas, uma pergunta central: o que separa quem vence no vôlei de praia profissional, e isso
mudou ao longo de duas décadas?

## 1. O ranking de entrada prediz o resultado da partida? Com que frequência a dupla pior ranqueada vence, e isso muda entre grupos, qualificatória e eliminatória?

In [0]:
SELECT coalesce(f.fase, 'total')                            AS fase,
       count(*)                                             AS partidas,
       round(count(p.flag_zebra) / count(*) * 100, 1)       AS pct_com_seed,
       round(avg(cast(p.flag_zebra AS int)) * 100, 1)       AS pct_zebra,
       round(100 - avg(cast(p.flag_zebra AS int)) * 100, 1) AS pct_favorito
FROM workspace.gold.fato_partida p
JOIN workspace.gold.dim_fase f USING (id_fase)
WHERE NOT p.flag_partida_incompleta
GROUP BY ROLLUP(f.fase)
ORDER BY CASE fase WHEN 'qualificatoria' THEN 1 WHEN 'grupos' THEN 2 WHEN 'eliminatoria' THEN 3 ELSE 4 END

**Resultado.** O ranking prediz o resultado. O favorito vence 70,4% das partidas, contra os 50% que se esperaria se o seed não contribuísse com nada, e a fase não muda isso, com 70,5% na qualificatória, 70,7% nos grupos e 70,3% na eliminatória. A variação de 0,4 ponto percentual entre as fases está dentro do ruído, então a ideia de que a eliminatória seria mais imprevisível não se sustenta.

## 2. O jogo mudou ao longo das duas décadas? Duração das partidas e proporção de 2×1 por circuito e por período.

In [0]:
%sql
SELECT t.circuito,
       d.ciclo_olimpico,
       min(d.ano)                                            AS de,
       max(d.ano)                                            AS ate,
       count(*)                                              AS partidas,
       round(avg(p.duracao_min), 1)                          AS duracao_media_min,
       round(avg(cast(p.total_sets = 3 AS int)) * 100, 1)    AS pct_tres_sets,
       round(avg(p.total_pontos), 1)                         AS pontos_por_partida,
       round(avg(p.duracao_min) / avg(p.total_pontos) * 60, 1) AS segundos_por_ponto
FROM workspace.gold.fato_partida p
JOIN workspace.gold.dim_torneio t USING (id_torneio)
JOIN workspace.gold.dim_data d USING (id_data)
WHERE NOT p.flag_partida_incompleta
  AND NOT p.flag_duracao_suspeita
  AND p.duracao_min IS NOT NULL
GROUP BY t.circuito, d.ciclo_olimpico
ORDER BY t.circuito, min(d.ano)

**Resultado.** O jogo mudou, mas só em um dos circuitos.

- **FIVB:** a partida média encurtou de 45,4 minutos no ciclo de Pequim para 39,0 no de Tóquio, com 84 a 85 pontos por partida em todos os ciclos. O que caiu foi o tempo por ponto, de 32,2 para 27,8 segundos, enquanto a proporção de 2×1 subiu de 31,5% para cerca de 34%.  As causas possíveis são ritmo de jogo, regras de intervalo ou critério de medição da duração, e o dado não distingue entre elas.
- **AVP:** nada se alterou. Duração entre 45 e 49 minutos e 2×1 entre 29% e 32% em todos os ciclos. O circuito não aparece em 2011 e 2012 porque faliu em agosto de 2010 e só voltou em 2012.
- **Comparação entre circuitos:** a diferença de duração era de 3 minutos no início da série e chega a 8 no fim.
- **Observação:** Sydney 2000 tem 0% de três sets e 104 segundos por ponto porque a partida era de set único até 15 com *side-out*, em que só quem sacava pontuava. O formato atual, de melhor de três sets a 21, entrou em 2001.

## 3. Quem é mais alto vence mais? A relação é a mesma no masculino e no feminino?

In [0]:
SELECT
  a.genero,
  FLOOR(a.altura_cm / 5) * 5                                     AS faixa_altura,
  COUNT(*)                                                       AS participacoes,
  ROUND(100 * AVG(CASE WHEN f.vencedor THEN 1 ELSE 0 END), 1)    AS taxa_vitoria
FROM workspace.gold.fato_atleta_partida f
JOIN workspace.gold.dim_atleta a USING (id_atleta)
WHERE a.altura_cm IS NOT NULL
GROUP BY 1, 2
HAVING COUNT(*) >= 1000
ORDER BY 1, 2

**Resultado.**  Sim, mas principalmente nas pontas. No masculino, a taxa de vitória aumenta com a altura: 37,9% na faixa de 175 cm, 43,4% na de 180 cm, e depois se estabiliza em um patamar de 50 a 51% entre 185 e 199 cm, intervalo que concentra três de cada quatro participações. Acima de 200 cm, a taxa volta a subir para 56,1% e chega a 59,2% na faixa de 205 cm. Entre a faixa mais baixa e a mais alta, a diferença chega a 21 pontos percentuais, a maior distância entre grupos de atletas encontrada neste trabalho.

No feminino, o padrão é semelhante, com um patamar intermediário mais amplo: entre 175 e 189 cm, faixa que concentra 80% das participações, a taxa de vitória permanece entre 50,2% e 51,8%. A faixa de 170 cm apresenta 45,9%, enquanto atletas com 190 cm ou mais chegam a 58,5%. A exceção é a faixa de 165 cm, com 50,6%; como reúne poucas participações (5 mil), é possível que essas atletas compensem a menor estatura com outros atributos para competir no circuito. A relação, portanto, aparece nos dois gêneros com direção e formato semelhantes; no masculino, a transição é mais longa porque a distribuição de alturas é mais espalhada.

Na prática, a altura parece oferecer vantagem principalmente nos extremos da distribuição: atletas muito altos apresentam taxas de vitória superiores, enquanto atletas muito baixos entram em desvantagem. Já dentro da faixa mais comum do circuito, a estatura pouco diferencia vencedores de perdedores.

## 4. A idade pesa no resultado? Em que faixa etária a taxa de vitória é maior?

In [0]:
SELECT
  LEAST(GREATEST(FLOOR(idade_na_partida / 5) * 5, 15), 40)      AS faixa_etaria,
  COUNT(*)                                                      AS participacoes,
  ROUND(100 * AVG(CASE WHEN vencedor THEN 1 ELSE 0 END), 1)     AS taxa_vitoria
FROM workspace.gold.fato_atleta_partida
WHERE idade_na_partida IS NOT NULL
  AND NOT flag_idade_atipica
GROUP BY 1
ORDER BY 1

**Resultado.** Pesa, mas pouco depois dos 25. A taxa de vitória sobe com a idade até os 30 e só cai depois
dos 40. A faixa mais numerosa é a de 25 a 29 anos (114 mil participações, 37% do total), mas quem vence mais
é a de 30 a 39: 35% das participações e a maior taxa de vitória. A vantagem, porém, é pequena: 2 pontos
percentuais sobre a faixa de 25 a 29, e as duas ficam a menos de 3 pontos dos 50% que seria o esperado caso
a idade não impactasse na vitória. Medida sobre mais de 100 mil participações, essa diferença não é ruído,
mas é pequena perto da distância entre os mais jovens e os demais: o menor de 20 anos vence 37% das vezes,
15 pontos abaixo da faixa de 35 a 39. Dessa forma, a idade não separa vencedores entre os adultos; separa
quem talvez ainda não tenha maturidade para o circuito

## 5. Jogar em casa ajuda? Taxa de vitória quando o torneio é no país do atleta, comparada a fora.

In [0]:
SELECT
  circuito,
  flag_em_casa                                                   AS em_casa,
  COUNT(*)                                                       AS participacoes,
  ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY circuito), 1) AS pct_participacoes,
  ROUND(100 * AVG(CASE WHEN vencedor THEN 1 ELSE 0 END), 1)      AS taxa_vitoria
FROM workspace.gold.fato_atleta_partida
JOIN workspace.gold.dim_torneio USING (id_torneio)
GROUP BY 1, 2
ORDER BY 1, 2 DESC

**Resultado.** Não. No AVP a pergunta quase não se aplica: 93,5% das participações são de atletas dos
Estados Unidos jogando nos Estados Unidos, e os dois lados vencem 50% (50,0 em casa, 50,7 fora). É no
FIVB, onde jogar em casa é a exceção (10,6% das participações), que a comparação vale, e o resultado é o
contrário do esperado: o atleta em casa vence 40,6% das partidas, contra 51,1% de quem joga fora. São 10,5
pontos percentuais contra o mandante.

Uma leitura possível é que o efeito venha da porta de entrada, não de nervosismo diante da torcida: o regulamento da FIVB
reserva vagas de convite para atletas do país-sede, que entram sem passar pelo ranking.

## 6. Nas partidas com estatística detalhada, qual fundamento mais separa vencedores de perdedores: ataque, saque, bloqueio ou defesa?

In [0]:
SELECT
  a.genero,
  f.vencedor,
  COUNT(*)                          AS participacoes,
  ROUND(AVG(f.tot_attacks), 1)      AS ataques,
  ROUND(AVG(f.tot_kills), 1)        AS pontos_ataque,
  ROUND(AVG(f.tot_hitpct), 2)       AS aproveitamento,
  ROUND(AVG(f.tot_errors), 1)       AS erros_ataque,
  ROUND(AVG(f.tot_aces), 2)         AS aces,
  ROUND(AVG(f.tot_serve_errors), 2) AS erros_saque,
  ROUND(AVG(f.tot_blocks), 2)       AS bloqueios,
  ROUND(AVG(f.tot_digs), 1)         AS defesas
FROM workspace.gold.fato_atleta_partida f
JOIN workspace.gold.dim_atleta a USING (id_atleta)
WHERE f.tem_estatistica
  AND NOT f.flag_estatistica_invalida
GROUP BY 1, 2
ORDER BY 1, 2 DESC

**Resultado.** O ataque, nos dois gêneros. Vencedores e perdedores atacam quase o mesmo tanto (25,1 contra 26,4 tentativas no masculino, 26,9 contra 27,4 no feminino), mas o vencedor converte mais (14,6 contra 13,0 pontos no masculino; 15,0 contra 12,4 no feminino) e erra menos (3,1 contra 4,7; 2,7 contra 4,0): o aproveitamento vai de 0,32 para 0,48 entre os homens e de 0,31 para 0,48 entre as mulheres. Somando pontos ganhos e erros evitados, são 3,2 pontos de diferença por atleta no masculino e 3,9 no feminino, mais que todos os outros fundamentos juntos. Bloqueio e ace têm a maior diferença proporcional (bloqueios 2,21 contra 1,36 e 1,26 contra 0,76; aces 1,10 contra 0,72 e 1,55 contra 0,92), mas acontecem pouco e somam pouco mais de um ponto por atleta nos dois gêneros; defesa fica no meio (7,3 contra 6,3; 9,6 contra 8,0) e erro de saque quase não separa (1,99 contra 2,19; 1,97 contra 1,95). O que muda entre gêneros é o volume de cada fundamento, não o que separa vencedor de perdedor: homens bloqueiam quase o dobro, mulheres fazem mais aces e defesas. Na prática, o que mais separa quem vence não é atacar mais nem sacar melhor, e sim errar menos no ataque.


## Discussão geral
A pergunta central era o que separa quem vence no vôlei de praia profissional e se isso mudou ao longo de duas décadas. As seis análises apontam para a mesma direção: dentro do circuito, a execução é o principal diferencial entre vencedores e perdedores, enquanto o biotipo exerce um efeito mais específico e concentrado nas extremidades da distribuição. O contexto, por sua vez, tem pouca influência sobre os resultados.

O biotipo pesa nas extremidades da distribuição. Homens abaixo de 1,80 m e mulheres abaixo de 1,75 m apresentam desvantagem, assim como os menores de 20 anos, que vencem apenas 37% das partidas, enquanto jogadores acima de 2,00 m e jogadoras acima de 1,90 m alcançam taxas de vitória próximas de 60%. Entre esses extremos, que concentram a grande maioria das participações, a taxa permanece próxima de 50%. Altura e idade, portanto, parecem funcionar mais como pré-requisito para estar no circuito do que como vantagem dentro dele; entre a maioria dos atletas, pouco diferenciam vencedores de perdedores.

A execução, por outro lado, aparece de forma consistente em diferentes recortes. O favorito pelo ranking vence cerca de sete em cada dez partidas, independentemente da fase, indicando uma hierarquia real e relativamente estável. Em quadra, o principal diferencial está no ataque: vencedores e perdedores atacam em volumes semelhantes, mas os vencedores cometem menos erros e convertem uma parcela maior das oportunidades. Bloqueios e aces também diferenciam os resultados, mas representam uma parcela menor da produção de pontos.

O contexto, por sua vez, tem pouco efeito aparente. Jogar em casa não produz uma vantagem relevante e, no circuito FIVB, a vaga destinada ao país-sede costuma ser ocupada por equipes que ainda não apresentam nível consolidado de circuito.

Por fim, o jogo mudou ao longo do período analisado, mas essa mudança aparece principalmente na FIVB. As partidas ficaram, em média, cerca de seis minutos mais curtas, apesar de manterem praticamente o mesmo número de pontos. O resultado é um jogo mais rápido. Assim, as demais respostas descrevem um jogo que, no fim do período, é mais rápido do que no início; verificar se as relações se mantêm quando medidas só no ciclo mais recente fica como trabalho futuro.